# 大语言模型 (LLM) 核心架构、微调与推理算法企业笔试手撕通关宝典
> **面向对象**：互联网大厂/AI 独角兽企业 大模型算法岗、LLM 预训练/微调/高并发推理面试手撕  
> **核心涵盖**：RMSNorm、SwiGLU 门控激活、RoPE 旋转位置编码、LoRA 线性层与权重动态融合、KV Cache 增量生成动力学、MQA/GQA 显存压缩、PagedAttention 页表映射、Temperature/Top-K/Top-P 组合采样  
> **设计准则**：纯 PyTorch 逐行白盒手撕，全面复现 LLaMA、DeepSeek、vLLM 核心工业基石。

---
### 核心模块速览
1. **模块一**：RMSNorm (均方根层归一化) 纯 PyTorch 手撕 (无减均值算子加速)
2. **模块二**：SwiGLU 门控前馈网络手撕 (现代主流开源大模型标配 FFN)
3. **模块三**：RoPE 旋转位置编码手撕 (二维复数旋转与相对内积保持)
4. **模块四**：LoRA 低秩自适应线性层手撕与 `merge()`/`unmerge()` 无损融合
5. **模块五**：KV Cache 动态自回归机制手撕 (Prefill 全量与 Decode 单步 O(1) 增量)
6. **模块六**：MQA 与 GQA 分组查询注意力机制手撕 (KV 头共享与广播)
7. **模块七**：PagedAttention 物理块页表映射模拟手撕 (Block Manager 分块注意力)
8. **模块八**：大模型推理解码策略手撕 (Temperature, Top-K, Top-P/核采样)

---
## 模块一：RMSNorm (均方根层归一化) 纯 PyTorch 手撕

### 【笔试考点与推导】
1. **公式**：
   $$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2 + \epsilon}} \odot \gamma$$
2. **与传统 LayerNorm 区别**：LayerNorm 需要计算均值 $\mu$ 并做中心化减均值 $x - \mu$；RMSNorm 假设输入的均值已基本为 0，直接除以二阶矩均方根（RMS），不仅节省了 $7\% \sim 10\%$ 的显存访问与计算时间，且保留了严格的尺度不变性。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # x: (..., dim)
        # 计算均方根: mean(x^2, dim=-1)
        variance = x.pow(2).mean(-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        return norm_x * self.weight

# 测试验证 RMSNorm
rmsnorm = RMSNorm(dim=64)
dummy_x = torch.randn(2, 4, 64) * 5.0 + 2.0
out_rms = rmsnorm(dummy_x)
# 验证输出的二阶矩接近 1
rms_val = torch.sqrt(out_rms.pow(2).mean(-1))
print("RMSNorm 归一化后各向量 RMS 均方根 (接近 1.0):", rms_val[0].detach().numpy())
assert torch.allclose(rms_val, torch.ones_like(rms_val), atol=1e-2)
print(">>> RMSNorm 均方根层归一化验证通过！")

RMSNorm 归一化后各向量 RMS 均方根 (接近 1.0): [0.99999994 1.         0.9999999  1.        ]
>>> RMSNorm 均方根层归一化验证通过！


---
## 模块二：SwiGLU 门控前馈网络手撕 (LLaMA 现代大模型标配 FFN)

### 【笔试考点与结构】
- **公式**：
  $$\text{SwiGLU}(x) = \left( \text{Swish}(x W_{\text{gate}}) \odot (x W_{\text{up}}) \right) W_{\text{down}}$$
  其中 $\text{Swish}(z) = z \cdot \sigma(z) = \text{SiLU}(z)$。
- **参数量等价折算**：为了与传统 $4d$ 的 FFN 拥有相同的总参数量，SwiGLU 的中间隐层维度通常取：
  $$d_{\text{ffn}} = \left\lfloor \frac{2}{3} \cdot 4d \right\rfloor = \left\lfloor \frac{8}{3} d \right\rfloor$$
  在 LLaMA 中还会向上取整为 256 或 128 的倍数以保证 GPU 硬件对齐。

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            # 8/3 倍扩展并对齐到 64 的倍数
            hidden_dim = int(2 * (4 * d_model) / 3)
            hidden_dim = ((hidden_dim + 63) // 64) * 64
            
        self.w_gate = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_up = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        # x: (B, L, d_model)
        gate = F.silu(self.w_gate(x))
        up = self.w_up(x)
        return self.w_down(gate * up)

swiglu = SwiGLU(d_model=64)
swi_out = swiglu(dummy_x)
print("SwiGLU 输出形状:", swi_out.shape)
assert swi_out.shape == dummy_x.shape
print(">>> SwiGLU 门控激活层验证通过！")

---
## 模块三：RoPE 旋转位置编码手撕 (二维复数旋转与相对内积保持)

### 【笔试顶级硬核考点】
1. **核心思想**：将向量相邻两两分组 $[x_1, x_2]$ 视为复数 $x_1 + i x_2$，乘上复数旋转因子 $e^{i m \theta} = \cos(m \theta) + i \sin(m \theta)$；
2. **二维旋转等价矩阵**：
   $$\begin{pmatrix} x_1' \\ x_2' \end{pmatrix} = \begin{pmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} = \begin{pmatrix} x_1 \cos(m\theta) - x_2 \sin(m\theta) \\ x_2 \cos(m\theta) + x_1 \sin(m\theta) \end{pmatrix}$$
3. **相对位置内积证明**：
   $$\langle R_m q, R_n k \rangle = q^T R_m^T R_n k = q^T R_{n-m} k = g(q, k, m-n)$$
   内积天然只依赖于相对距离 $m-n$！

In [ ]:
def precompute_rope_freqs(dim, max_len=1000, base=10000.0):
    """预计算旋转角度的 cos 和 sin 表"""
    # dim 必须为偶数
    half_dim = dim // 2
    theta = 1.0 / (base ** (torch.arange(0, half_dim).float() / half_dim)) # (dim/2,)
    t = torch.arange(max_len).float()                                      # (max_len,)
    freqs = torch.outer(t, theta)                                          # (max_len, dim/2)
    cos = torch.cos(freqs) # (max_len, dim/2)
    sin = torch.sin(freqs) # (max_len, dim/2)
    return cos, sin

def apply_rope(x, cos, sin):
    """
    x: (B, seq_len, num_heads, head_dim)
    """
    seq_len = x.size(1)
    head_dim = x.size(-1)
    half_dim = head_dim // 2
    
    # 拆分两两成对维度: (B, L, H, dim/2)
    x1 = x[..., :half_dim]
    x2 = x[..., half_dim:]
    
    # cos, sin 调整为广播形状: (1, L, 1, dim/2)
    c = cos[:seq_len, :].unsqueeze(0).unsqueeze(2)
    s = sin[:seq_len, :].unsqueeze(0).unsqueeze(2)
    
    # 旋转矩阵计算: [x1*cos - x2*sin ; x1*sin + x2*cos]
    rx1 = x1 * c - x2 * s
    rx2 = x1 * s + x2 * c
    return torch.cat([rx1, rx2], dim=-1)

# 测试验证 RoPE 相对位置内积恒等性
cos_tab, sin_tab = precompute_rope_freqs(dim=32, max_len=100)
q = torch.randn(1, 1, 1, 32) # 位置 m=5
k = torch.randn(1, 1, 1, 32) # 位置 n=2 (相对距离 3)

# 模拟位置 5 的 q 与位置 2 的 k
q_rot_5 = apply_rope(q, cos_tab[5:], sin_tab[5:])
k_rot_2 = apply_rope(k, cos_tab[2:], sin_tab[2:])
inner_prod_1 = torch.sum(q_rot_5 * k_rot_2)

# 模拟位置 8 的 q 与位置 5 的 k (相对距离同为 3)
q_rot_8 = apply_rope(q, cos_tab[8:], sin_tab[8:])
k_rot_5 = apply_rope(k, cos_tab[5:], sin_tab[5:])
inner_prod_2 = torch.sum(q_rot_8 * k_rot_5)

print(f"相对距离为 3 时的内积 1: {inner_prod_1.item():.5f}")
print(f"相对距离为 3 时的内积 2: {inner_prod_2.item():.5f}")
assert torch.isclose(inner_prod_1, inner_prod_2, atol=1e-4)
print(">>> RoPE 相对位置内积不变性严格证明通过！")

---
## 模块四：LoRA 低秩自适应线性层手撕与 `merge()`/`unmerge()` 无损融合

### 【笔试考点与公式】
- **公式**：
  $$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} B A x$$
  其中 $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$（冻结参数），$A \in \mathbb{R}^{r \times d_{\text{in}}}$（高斯初始化），$B \in \mathbb{R}^{d_{\text{out}} \times r}$（全 0 初始化，确保初始状态 $\Delta W = 0$）；
- **推理部署融合 (Weight Merging)**：
  $$W_{\text{merged}} = W_0 + \frac{\alpha}{r} B A$$
  将旁路矩阵直接加到主权重中，推理延迟零增加！

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=8, lora_alpha=16.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.scaling = lora_alpha / r
        
        # 冻结原始预训练全量权重
        self.weight = nn.Parameter(torch.randn(out_features, in_features), requires_grad=False)
        
        # LoRA 旁路低秩矩阵 A 和 B
        self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, r)) # 初始全 0
        self.merged = False

    def forward(self, x):
        if self.merged:
            return F.linear(x, self.weight)
        else:
            # 原始前向 + 旁路缩放低秩前向
            base_out = F.linear(x, self.weight)
            lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scaling
            return base_out + lora_out

    def merge(self):
        """将 LoRA 权重融入主干，消除推理开销"""
        if not self.merged:
            # W = W + scaling * (B @ A)
            self.weight.data += self.scaling * (self.lora_B @ self.lora_A)
            self.merged = True

    def unmerge(self):
        """撤销融合"""
        if self.merged:
            self.weight.data -= self.scaling * (self.lora_B @ self.lora_A)
            self.merged = False

# 测试 LoRA
lora = LoRALinear(in_features=32, out_features=16, r=4, lora_alpha=8.0)
x_test = torch.randn(2, 32)
out_before = lora(x_test)
lora.merge()
out_after = lora(x_test)
assert torch.allclose(out_before, out_after, atol=1e-5)
print(">>> LoRA 动态 forward 与 merge() 权重无损融合验证成功！")

---
## 模块五：KV Cache 动态自回归机制手撕 (Prefill 与 Decode 两阶段)

### 【笔试高频考点】
1. **Prefill (首字计算阶段)**：输入 Prompt 序列长为 $L$，全并行计算注意力，将生成的 Key 和 Value 缓存入 `k_cache` 和 `v_cache`；
2. **Decode (逐字生成阶段)**：每一步只输入 **单个 Token**（$q$ 长度为 1），仅计算当前 Token 的 $k_{\text{new}}, v_{\text{new}}$ 并拼入 Cache，复杂度由 $O(L^2)$ 骤降为 **$O(L)$**！

In [ ]:
class KVCacheAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, kv_cache=None):
        """
        x: (B, 1, d_model) 或 (B, L, d_model)
        kv_cache: 元组 (k_cache, v_cache)
        """
        B, L, D = x.shape
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)
        
        if kv_cache is not None:
            prev_k, prev_v = kv_cache
            # 增量拼接已缓存的历史键值
            k = torch.cat([prev_k, k], dim=1) # (B, L_prev + L, D)
            v = torch.cat([prev_v, v], dim=1)
            
        new_cache = (k, v)
        
        # 计算当前步注意力
        scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)
        return out, new_cache

# 测试 Prefill 与 Decode
kv_attn = KVCacheAttention(d_model=16)
prompt = torch.randn(1, 4, 16) # Prefill: 4 个 token
out_prefill, cache = kv_attn(prompt)
print("Prefill 完成，当前 KV Cache 长度:", cache[0].shape[1])

# Decode 步: 每次仅输入 1 个 token
next_tok = torch.randn(1, 1, 16)
out_decode, cache = kv_attn(next_tok, kv_cache=cache)
print("Decode 第 1 步完成，新 KV Cache 长度:", cache[0].shape[1])
assert cache[0].shape == (1, 5, 16)
print(">>> KV Cache 两阶段自回归增量生成验证通过！")

---
## 模块六：MQA 与 GQA 分组查询注意力机制手撕 (显存极致压缩)

### 【笔试必问八股与手撕】
- **MHA (多头注意力)**：$N$ 个 Query 头对应 $N$ 个独立的 Key/Value 头；
- **MQA (Multi-Query Attention)**：$N$ 个 Query 头**共享唯独 1 个** Key/Value 头（显存极度省，但表征能力微损）；
- **GQA (Grouped-Query Attention)**：折中艺术（LLaMA-2/3 标配），将 $N$ 个 Query 头分为 $G$ 组，每组共享 1 个 Key/Value 头。
- **手撕关键**：使用 `torch.repeat_interleave` 将 Key 和 Value 头广播拓展回 Query 头数！

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_q_heads=8, num_kv_heads=2):
        super().__init__()
        assert num_q_heads % num_kv_heads == 0
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.num_groups = num_q_heads // num_kv_heads # 每组拥有的 Q 头数
        self.head_dim = d_model // num_q_heads
        
        self.W_q = nn.Linear(d_model, num_q_heads * self.head_dim)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.head_dim) # 压缩参数
        self.W_v = nn.Linear(d_model, num_kv_heads * self.head_dim)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, L, _ = x.shape
        # Q: (B, num_q_heads, L, D)
        q = self.W_q(x).view(B, L, self.num_q_heads, self.head_dim).transpose(1, 2)
        # K, V: (B, num_kv_heads, L, D)
        k = self.W_k(x).view(B, L, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, L, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        # 核心广播扩展: 将 K, V 在 head 维度复制 num_groups 次与 Q 对齐
        k_expanded = torch.repeat_interleave(k, repeats=self.num_groups, dim=1) # (B, num_q_heads, L, D)
        v_expanded = torch.repeat_interleave(v, repeats=self.num_groups, dim=1)
        
        scores = torch.matmul(q, k_expanded.transpose(-2, -1)) / (self.head_dim ** 0.5)
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v_expanded)
        
        out = out.transpose(1, 2).contiguous().view(B, L, -1)
        return self.W_o(out)

# 验证 GQA
gqa = GroupedQueryAttention(d_model=64, num_q_heads=8, num_kv_heads=2)
out_gqa = gqa(dummy_x)
print("GQA 输出形状:", out_gqa.shape)
assert out_gqa.shape == (2, 4, 64)
print(">>> GQA 分组查询注意力机制验证通过！")

---
## 模块七：PagedAttention 物理块页表映射模拟手撕 (vLLM 核心基石)

### 【笔试考点与面试原理解析】
1. **传统 KV Cache 痛点**：必须预先分配连续大块虚拟显存，导致严重的显存碎片化与浪费；
2. **PagedAttention 核心机制**：借用操作系统分页 (Virtual Memory Paging) 思想，将 KV Cache 切分为固定大小的 Block（如 16 个 Token 一页），通过 **页表 (Block Table)** 将逻辑连续的 Token 映射到物理离散的 Block。

In [ ]:
class SimplePagedKVCacheManager:
    """PagedAttention 核心物理块内存管理器仿真"""
    def __init__(self, block_size=4, num_blocks=8, head_dim=16):
        self.block_size = block_size
        self.num_blocks = num_blocks
        # 物理显存显式分配池: (num_blocks, block_size, head_dim)
        self.physical_k_blocks = torch.zeros(num_blocks, block_size, head_dim)
        self.free_blocks = list(range(num_blocks))
        self.block_tables = {} # seq_id -> [block_idx_0, block_idx_1, ...]

    def allocate_for_token(self, seq_id, token_idx, k_vector):
        """将新生成的 token 写入逻辑页表对应的物理块中"""
        if seq_id not in self.block_tables:
            self.block_tables[seq_id] = []
            
        block_idx_in_seq = token_idx // self.block_size
        offset = token_idx % self.block_size
        
        # 若需要分配新物理块
        if block_idx_in_seq >= len(self.block_tables[seq_id]):
            assert self.free_blocks, "显存物理块耗尽 (OOM)!"
            new_blk = self.free_blocks.pop(0)
            self.block_tables[seq_id].append(new_blk)
            
        phys_blk_id = self.block_tables[seq_id][block_idx_in_seq]
        self.physical_k_blocks[phys_blk_id, offset] = k_vector

    def get_logical_k_tensor(self, seq_id, seq_len):
        """根据页表将离散块拼接还原出完整的逻辑序列"""
        k_list = []
        for i in range(seq_len):
            blk_id = self.block_tables[seq_id][i // self.block_size]
            offset = i % self.block_size
            k_list.append(self.physical_k_blocks[blk_id, offset])
        return torch.stack(k_list)

# 测试 PagedAttention 内存管理
paged_mgr = SimplePagedKVCacheManager(block_size=2, num_blocks=4, head_dim=4)
# 写入序列 0 的 3 个 token (跨越 2 个物理页)
paged_mgr.allocate_for_token(seq_id=0, token_idx=0, k_vector=torch.tensor([1., 1., 1., 1.]))
paged_mgr.allocate_for_token(seq_id=0, token_idx=1, k_vector=torch.tensor([2., 2., 2., 2.]))
paged_mgr.allocate_for_token(seq_id=0, token_idx=2, k_vector=torch.tensor([3., 3., 3., 3.]))

recovered_k = paged_mgr.get_logical_k_tensor(seq_id=0, seq_len=3)
print("通过页表正确拼接的逻辑连续 Key 矩阵:\n", recovered_k.numpy())
assert recovered_k.shape == (3, 4)
print(">>> PagedAttention 内存分页机制验证成功！")

---
## 模块八：大模型推理解码策略手撕 (Temperature, Top-K, Top-P 组合采样)

### 【笔试考题与采样全流程】
1. **Temperature**：$z' = z / T$ 缩放平滑概率；
2. **Top-K 截断**：保留最大的 $K$ 个 Logits，其余填入 $-\infty$；
3. **Top-P (核采样)**：按概率从大到小排序累加，截取累加和刚好超过 $P$ 的最小集合，其余填入 $-\infty$；
4. **Softmax 采样**：从过滤后的分布中随机抽样。

In [ ]:
def top_k_top_p_sampling(logits, temperature=1.0, top_k=0, top_p=0.9):
    """
    大模型推理解码标准采样器
    """
    # 1. 温度缩放
    logits = logits / max(temperature, 1e-5)
    
    # 2. Top-K 过滤
    if top_k > 0:
        top_k = min(top_k, logits.size(-1))
        # 找出第 k 大的值作为阈值
        val, _ = torch.topk(logits, top_k)
        threshold = val[-1]
        logits[logits < threshold] = -float('Inf')
        
    # 3. Top-P (核采样) 过滤
    if top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        
        # 剔除累计概率超过 top_p 的多余 token
        sorted_indices_to_remove = cumulative_probs > top_p
        # 保证至少保留第 1 个 token
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = False
        
        # 还原回原序列并置为 -inf
        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[indices_to_remove] = -float('Inf')
        
    # 4. 采样
    probs = F.softmax(logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    return next_token.item()

# 测试组合采样
test_logits = torch.tensor([1.0, 2.0, 10.0, 3.0, 0.5, -2.0])
token_sampled = top_k_top_p_sampling(test_logits, temperature=0.7, top_k=3, top_p=0.95)
print("组合采样出的 Token ID:", token_sampled)
assert token_sampled in [1, 2, 3] # 应大概率落在高分区域
print(">>> 大模型推理解码策略组合验证通过！")

---
## 企业笔试手撕核心口诀与雷区速记卡

```
1. RMSNorm 极速归一: 省略减均值，除以 sqrt(mean(x^2)+eps) * weight，省时抗溢出。
2. SwiGLU 门控前馈: silu(gate) * up 后过 down，中间隐层 8/3 维度倍数对齐硬件。
3. RoPE 二维复数转: [x1*cos - x2*sin ; x1*sin + x2*cos]，内积天然只取决于相对位置差。
4. LoRA 部署无损融: A 用高斯初始化，B 用全 0 初始化，推理时 W + (alpha/r)*(B@A) 一键融合。
5. GQA 广播扩展: K, V 用 repeat_interleave 复制扩展与 Q 对齐，极致削减 KV Cache 显存开销。
```